In [39]:
import os, json
import requests
from rich import print as rprint
from dotenv import load_dotenv

BASE_URL = 'http://127.0.0.1:8000'
load_dotenv()

True

## 1. Analýza životopisu s cílovým povoláním
`POST /resume` — nahraje soubor + volitelný `target_job`
→ `SuggestionResponse` (`top_suggestion`, `target_suggestion`)

In [107]:
resume_path = os.path.join('testfiles', 'Životopis.pdf')
target_job = 'zlatokop'

with open(resume_path, 'rb') as f:
    files = {'file': (os.path.basename(resume_path), f)}
    data = {'target_job': target_job}

    r = requests.post(f'{BASE_URL}/resume', files=files, data=data)

In [104]:
res = r.json()

missing = set([s['label'] for s in res['top_suggestion']['missing_skills']])
best = res['top_suggestion']['occupation']
lbl = best['label']

print(
    f'nejlepší skore: {lbl} ({best['score']})\n\n'
    f'další dovednosti k {lbl}:\n{'\n'.join(missing)}'
)


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [29]:
missing_ = set([s['label'] for s in res['target_suggestion']['missing_skills']])
goal = res['target_suggestion']['occupation']
lbl_ = goal['label']
print(
    f'skore zadané pozice z dovedností: {round(goal['score'],5)} ({lbl_})\n\n'
    f'další dovednosti k {lbl_}:\n{'\n'.join(missing)}'
)

skore zadané pozice z dovedností: 0.70614 (projektant/projektantka)

další dovednosti k projektant/projektantka:
techniky ručního kreslení
obvodové schéma
mechanika motorových vozidel
plánovat výrobní procesy
navádění, navigace a řízení
používat technické komunikační dovednosti
technologie stealth
odhadovat náklady na stavební materiály
topografie
vypracovat kusovník
součásti topení, ventilace, klimatizace a chlazení
elektrické stroje
provést kalkulaci materiálu pro stavbu zařízení
elektřina
součásti klimatizačních systémů
vypracovávat stavební dokumentaci
mechanika tekutin
syntetické přírodní prostředí
technické zásady
uchovávat záznamy o postupu prací
strojírenství
školit zaměstnance
číst technické výkresy
strojírenské principy
modelovat elektrické systémy
navrhovat prototypy
používat měřicí přístroj
mechanika materiálů
průmyslové inženýrství
poskytovat poradenství v oblasti stavebních otázek
připravovat zprávy s analýzou nákladů a výnosů
kontrolovat dodržování předpisů pro železničn

## 2. Analýza životopisu → práce z různých oblastí pracovního trhu
`POST /resume/domains` — nahraje soubor
→ `DomainResponse` (`domains`: list of `DomainResult`, each with `domain`, `occ_count`, `occupations`)

In [109]:
resume_path = os.path.join('testfiles', 'Životopis.pdf')

with open(resume_path, 'rb') as f:
    files = {'file': (os.path.basename(resume_path), f)}
    r = requests.post(f'{BASE_URL}/resume/domains', files=files)


In [32]:
rprint(r.json()) #TODO fix

{'domains': []}


## 3. Dovednosti → povolání dle skóre
`POST /text` — JSON s klíčem `skills` (čárkou oddělené dovednosti) + volitelný `target_job`
→ `SkillsResponse` (`suggestions`: list of `Suggestion`)

In [40]:
payload = {
    'skills': 'Python, machine learning, analýza dat, SQL, statistika',
    'target_job': 'Business analytik'
}
r = requests.post(f'{BASE_URL}/text', json=payload)
print(f'Status: {r.status_code}')

Status: 200


In [ ]:
rprint(r.json())

## 4. Skóre zadaného povolání
`POST /text/goal` — JSON s `skills` + `target_job`
→ `TargetJobResponse` (`top_suggestion`, `target_suggestion`)

In [42]:
payload = {
    'skills': 'Python, machine learning, analýza dat, SQL, statistika, matematické modelování, práce s databázemi',
    'target_job': 'datový analytik'
}
r = requests.post(f'{BASE_URL}/text/goal', json=payload)

print(f'Status: {r.status_code}')

Status: 200


In [ ]:
rprint(r.json())

## 5. Vyhledávání samotných entit
`POST /query` — JSON s `text`, volitelný `entity_type` (`all`/`occupation`/`skill`/`skill_group`/`isco_group`), volitelný `n`
→ `List[EntityResult]`

In [98]:
query = {
    'text': 'referentka nákladní dopravy',
    'entity_type': 'occupation',
    'n': 5
}
r = requests.post(f'{BASE_URL}/query', json=query)

print(f'Status: {r.status_code}')

Status: 200


In [99]:
rprint(r.json())

[
    {
        'id': 3248,
        'cosine_score': 0.7707703113555908,
        'entity_type': 'occupation',
        'esco_uri': 'http://data.europa.eu/esco/occupation/49e28648-4fa8-4b36-ac4d-781c108547bf',
        'label': 'dispečer nákladní dopravy/dispečerka nákladní dopravy',
        'code': '4323.6',
        'isco_code': 4323.0,
        'description': 'Dispečeři nákladní dopravy přijímají a předávají spolehlivé zprávy, sledují vozidla a 
zařízení a zaznamenávají další důležité informace. Dohlížejí na plánovací operace při koordinaci různých druhů 
dopravy. Dispečeři nákladní dopravy strukturují trasy nebo služby nákladní dopravy a určují vhodný způsob dopravy. 
Jsou rovněž odpovědní za údržbu zařízení a vozidel a vysílání pracovníků. Dispečeři nákladní dopravy poskytují 
právní a smluvní dokumentaci pro účastníky přepravy.'
    },
    {
        'id': 3249,
        'cosine_score': 0.7698688507080078,
        'entity_type': 'occupation',
        'esco_uri': 'http://data.europa.eu/esco/occupation/49e28648-4fa8-4b36-ac4d-781c108547bf',
        'label': 'dispečer nákladní dopravy/dispečerka nákladní dopravy',
        'code': '4323.6',
        'isco_code': 4323.0,
        'description': 'Dispečeři nákladní dopravy přijímají a předávají spolehlivé zprávy, sledují vozidla a 
zařízení a zaznamenávají další důležité informace. Dohlížejí na plánovací operace při koordinaci různých druhů 
dopravy. Dispečeři nákladní dopravy strukturují trasy nebo služby nákladní dopravy a určují vhodný způsob dopravy. 
Jsou rovněž odpovědní za údržbu zařízení a vozidel a vysílání pracovníků. Dispečeři nákladní dopravy poskytují 
právní a smluvní dokumentaci pro účastníky přepravy.'
    },
    {
        'id': 3030,
        'cosine_score': 0.740752637386322,
        'entity_type': 'occupation',
        'esco_uri': 'http://data.europa.eu/esco/occupation/45037d43-a8f5-4f46-b332-b2935bc305f4',
        'label': 'řidič nákladního automobilu/řidička nákladního automobilu',
        'code': '8332.2',
        'isco_code': 8332.0,
        'description': 'Řidiči nákladních automobilů obsluhují vozidla, jako jsou nákladní vozidla a dodávky. Mohou
mít rovněž na starost nakládku a vykládku nákladu.'
    },
    {
        'id': 3033,
        'cosine_score': 0.7326273322105408,
        'entity_type': 'occupation',
        'esco_uri': 'http://data.europa.eu/esco/occupation/45037d43-a8f5-4f46-b332-b2935bc305f4',
        'label': 'řidič nákladního automobilu/řidička nákladního automobilu',
        'code': '8332.2',
        'isco_code': 8332.0,
        'description': 'Řidiči nákladních automobilů obsluhují vozidla, jako jsou nákladní vozidla a dodávky. Mohou
mít rovněž na starost nakládku a vykládku nákladu.'
    },
    {
        'id': 3032,
        'cosine_score': 0.7150541543960571,
        'entity_type': 'occupation',
        'esco_uri': 'http://data.europa.eu/esco/occupation/45037d43-a8f5-4f46-b332-b2935bc305f4',
        'label': 'řidič nákladního automobilu/řidička nákladního automobilu',
        'code': '8332.2',
        'isco_code': 8332.0,
        'description': 'Řidiči nákladních automobilů obsluhují vozidla, jako jsou nákladní vozidla a dodávky. Mohou
mít rovněž na starost nakládku a vykládku nákladu.'
    }
]

### Vyhledávání dovedností

In [96]:
query = {
    'text': 'míchání nealkoholických cocktailů',
    'entity_type': 'skill',
    'n': 5
}
r = requests.post(f'{BASE_URL}/query', json=query)

print(f'Status: {r.status_code}')

Status: 200


In [97]:
rprint(r.json())

[
    {
        'id': 8409,
        'cosine_score': 0.8717737793922424,
        'entity_type': 'skill',
        'esco_uri': 'http://data.europa.eu/esco/skill/81d5b408-e805-4788-8dbd-42f22e8fd199',
        'label': 'připravovat míchané nápoje',
        'code': '',
        'isco_code': '',
        'description': 'Vytvořit řadu míchaných alkoholických nápojů, jako jsou koktejly a long drinky a 
nealkoholické nápoje, podle receptů.'
    },
    {
        'id': 8412,
        'cosine_score': 0.8701602220535278,
        'entity_type': 'skill',
        'esco_uri': 'http://data.europa.eu/esco/skill/81d5b408-e805-4788-8dbd-42f22e8fd199',
        'label': 'připravovat míchané nápoje',
        'code': '',
        'isco_code': '',
        'description': 'Vytvořit řadu míchaných alkoholických nápojů, jako jsou koktejly a long drinky a 
nealkoholické nápoje, podle receptů.'
    },
    {
        'id': 8411,
        'cosine_score': 0.8644611239433289,
        'entity_type': 'skill',
        'esco_uri': 'http://data.europa.eu/esco/skill/81d5b408-e805-4788-8dbd-42f22e8fd199',
        'label': 'připravovat míchané nápoje',
        'code': '',
        'isco_code': '',
        'description': 'Vytvořit řadu míchaných alkoholických nápojů, jako jsou koktejly a long drinky a 
nealkoholické nápoje, podle receptů.'
    },
    {
        'id': 8415,
        'cosine_score': 0.8621479272842407,
        'entity_type': 'skill',
        'esco_uri': 'http://data.europa.eu/esco/skill/81d5b408-e805-4788-8dbd-42f22e8fd199',
        'label': 'připravovat míchané nápoje',
        'code': '',
        'isco_code': '',
        'description': 'Vytvořit řadu míchaných alkoholických nápojů, jako jsou koktejly a long drinky a 
nealkoholické nápoje, podle receptů.'
    },
    {
        'id': 8413,
        'cosine_score': 0.8407094478607178,
        'entity_type': 'skill',
        'esco_uri': 'http://data.europa.eu/esco/skill/81d5b408-e805-4788-8dbd-42f22e8fd199',
        'label': 'připravovat míchané nápoje',
        'code': '',
        'isco_code': '',
        'description': 'Vytvořit řadu míchaných alkoholických nápojů, jako jsou koktejly a long drinky a 
nealkoholické nápoje, podle receptů.'
    }
]

### Vyhledávání ve všech entitách
občas to zvládne i dost niche věci

In [78]:
query = {
    'text': 'koordinace dobrovolníků na festivalu nerdů',
    'entity_type': 'all',
    'n': 20
}
r = requests.post(f'{BASE_URL}/query', json=query)

print(f'Status: {r.status_code}')

Status: 200


In [79]:
rprint(r.json())


[
    {
        'id': 13792,
        'cosine_score': 0.6550639867782593,
        'entity_type': 'skill',
        'esco_uri': 'http://data.europa.eu/esco/skill/2d667669-e738-41d2-8b6d-ff53fa952564',
        'label': 'zapojovat dobrovolníky',
        'code': '',
        'isco_code': '',
        'description': 'Přijímat, motivovat a řídit dobrovolníky v organizaci nebo v útvaru organizace. Řídit 
vztahy s dobrovolníky před tím, než se ujmou dobrovolnické činnosti, a následně po celou dobu jejího výkonu v rámci
organizace až do doby po ukončení formální dohody o poskytování dobrovolnické činnosti.'
    }
]